# Random Forest para depressão (NHIS 2025): binário e multinível, com e sem ansiedade


## Hipótese

> É possível prever depressão a partir de dados de saúde física e do contexto de vida, **sem** depender de sintomas de transtornos mentais.

Esperamos que o desempenho sem sintomas mentais seja **pior** do que com eles. A hipótese não exige que sejam iguais: ela se sustenta se o modelo ainda tiver poder preditivo real, bem acima do acaso.

## Cenários

1. **Cenário A (exploratório, com ansiedade):** removemos de $X$ apenas os sintomas de depressão (PHQ) e **mantemos** as variáveis de ansiedade (GAD-7). Serve para explorar os dados e como referência de desempenho máximo.
2. **Cenário B (hipótese, sem sintomas mentais):** removemos de $X$ os sintomas de depressão **e** de ansiedade. Sobram variáveis sociodemográficas, de saúde física, de acesso a serviços de saúde e de contexto social.

## Roteiro

1. Preparação dos dados e definição dos cenários.
2. Função de treino e avaliação (alvo binário).
3. Cenário A e Cenário B, comparativo e variáveis mais importantes.
4. **Por que usar o alvo binário** (comparação com os 4 níveis de severidade do PHQ).
5. Ajuste de hiperparâmetros (`RandomizedSearchCV`) no cenário B, binário e multinível.
6. Perfil médio por nível de severidade.
7. Teste de sensibilidade: o resultado depende das variáveis cognitivas e de solidão?
8. Priorizando o recall: rastreio em saúde.
9. O que mais melhora o recall no Random Forest.
10. Interpretação e conclusão.

## Desenho do experimento (para a comparação ser justa)

- Mesma base (`adult25_limpo.csv`), mesmo split treino/teste (80/20 estratificado, `random_state=42`) em todos os modelos.
- **Mesmas linhas** nos dois cenários: registros sem `-1` (não resposta) nas variáveis do cenário A, o mais restritivo. Nos notebooks antigos, manter ou remover o `-1` de $X$ mudou pouco o resultado (acurácia de 0,77 contra 0,78 no multinível), por isso essa escolha foi unificada: usamos só registros completos.
- Mesmo modelo e mesmos hiperparâmetros-base nos dois cenários. Só muda o conjunto de colunas.
- **O `PESO_AMOSTRAL` não é usado no treino.** O desbalanceamento (cerca de 21% de casos positivos) é tratado com `class_weight='balanced'`. O peso entra apenas no cálculo das métricas de avaliação, e as matrizes de confusão são sempre em contagem de registros.
- **Por que priorizar o recall:** em saúde, os dois erros não custam o mesmo. Um falso negativo é uma pessoa com depressão que o modelo não identifica e que, portanto, não é encaminhada para avaliação. Um falso positivo é uma pessoa sem depressão que passa por uma avaliação a mais. Em um rastreio, é preferível encontrar mais casos, mesmo com mais falsos positivos. Por isso, além das métricas de sempre, as seções 10 e 11 mostram quanto recall é possível ganhar e quanto isso custa em falsos positivos.

# 1. Preparação dos dados e definição dos cenários

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

PESO_COLUNA = 'PESO_AMOSTRAL'
TARGET_ORIGINAL = 'SEVERIDADE_DEPRESSAO_PHQ'
TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
RANDOM_STATE = 42

# Sintomas diretos de depressão (PHQ): removidos nos DOIS cenários
COLUNAS_DEPRESSAO = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ'
]

# Ansiedade (GAD-7): mantida no cenário A, removida no cenário B
COLUNAS_ANSIEDADE = [
    'HISTORICO_TRANSTORNO_ANSIEDADE', 'MED_ANSIEDADE',
    'FREQ_ANSIEDADE', 'INTENSIDADE_ANSIEDADE',
    'FREQ_ANSIEDADE_2SEM', 'FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM',
    'FREQ_PREOCUPACAO_DESNECESSARIA_2SEM', 'FREQ_DIFICULDADE_DESCANSAR_2SEM',
    'FREQ_AGITACAO_INQUIETA', 'FREQ_IRRITABILIDADE_2SEM',
    'FREQ_MEDO_2SEM', 'SEVERIDADE_ANSIEDADE'
]

# Carregar a base limpa
df = pd.read_csv(Path('../../01 Preparação dos dados/dados/limpos') / 'adult25_limpo.csv').set_index('ID_DOMICILIO')

# Garante que nenhuma coluna foi digitada errado (o drop abaixo ignoraria erros silenciosamente)
ausentes = [c for c in COLUNAS_DEPRESSAO + COLUNAS_ANSIEDADE if c not in df.columns]
assert not ausentes, f'Colunas inexistentes na base: {ausentes}'

# Remover registros indefinidos no alvo (-1 e 5) e binarizar: 0 = sem depressão | 1 = com depressão
df = df[~df[TARGET_ORIGINAL].isin([-1, 5])].copy()
df[TARGET_BIN] = (df[TARGET_ORIGINAL] != 1).astype(int)

# Definição dos dois cenários: o que é removido de X em cada um
CENARIOS = {
    'A: com ansiedade (exploratório)': COLUNAS_DEPRESSAO,
    'B: sem sintomas mentais (hipótese)': COLUNAS_DEPRESSAO + COLUNAS_ANSIEDADE,
}
BASE_REMOVER = [TARGET_BIN, PESO_COLUNA]

# Mesmas linhas nos dois cenários: registros sem -1 nas variáveis do cenário A (o mais restritivo)
X_cenario_a = df.drop(columns=BASE_REMOVER + CENARIOS['A: com ansiedade (exploratório)'])
mascara_completos = (X_cenario_a != -1).all(axis=1).to_numpy()
df_modelo = df[mascara_completos].copy()

# Mesmo split nos dois cenários (por posição).
# Só existem DUAS partições: treino (80%) e teste (20%). Não há conjunto de validação separado:
# a validação é a validação cruzada feita DENTRO do treino (hiperparâmetros e limiar). O teste só mede o resultado.
posicoes = np.arange(len(df_modelo))
pos_tr, pos_te = train_test_split(
    posicoes, test_size=0.2, random_state=RANDOM_STATE, stratify=df_modelo[TARGET_BIN]
)

y_tr = df_modelo[TARGET_BIN].iloc[pos_tr]
y_te = df_modelo[TARGET_BIN].iloc[pos_te]
w_te = df_modelo[PESO_COLUNA].iloc[pos_te]

print(f'Registros no experimento: {len(df_modelo)} (treino: {len(pos_tr)} | teste: {len(pos_te)})')
print(f'Proporção de depressão (ponderada pelo peso amostral): {np.average(df_modelo[TARGET_BIN], weights=df_modelo[PESO_COLUNA]):.1%}')
for nome, remover in CENARIOS.items():
    n_feat = df_modelo.drop(columns=BASE_REMOVER + remover).shape[1]
    print(f'Cenário {nome}: {n_feat} variáveis preditoras')

Registros no experimento: 22084 (treino: 17667 | teste: 4417)
Proporção de depressão (ponderada pelo peso amostral): 21.2%
Cenário A: com ansiedade (exploratório): 58 variáveis preditoras
Cenário B: sem sintomas mentais (hipótese): 46 variáveis preditoras


# 2. Função de treino e avaliação

A mesma função é usada nos dois cenários, garantindo que a **única diferença** entre eles seja o conjunto de variáveis. Retorna as métricas da classe positiva (com depressão), ponderadas pelo `PESO_AMOSTRAL`, a matriz de confusão em **contagem de registros reais** do teste (sem peso) e a importância das variáveis do modelo.

In [2]:
def treinar_avaliar(colunas_remover):
    X = df_modelo.drop(columns=BASE_REMOVER + colunas_remover)
    X_tr, X_te = X.iloc[pos_tr], X.iloc[pos_te]

    num_cols = [c for c in ['RAZAO_RENDA_POBREZA'] if c in X.columns]
    cat_cols = [c for c in X.columns if c not in num_cols]
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ])
    X_tr_proc = preprocessor.fit_transform(X_tr)
    X_te_proc = preprocessor.transform(X_te)

    modelo = RandomForestClassifier(
        n_estimators=300, max_depth=25, min_samples_split=5,
        max_features='log2', class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    modelo.fit(X_tr_proc, y_tr)

    y_pred = modelo.predict(X_te_proc)
    y_proba = modelo.predict_proba(X_te_proc)[:, 1]

    metricas = {
        'Nº de variáveis': X.shape[1],
        'ROC-AUC': roc_auc_score(y_te, y_proba, sample_weight=w_te),
        'Acurácia': accuracy_score(y_te, y_pred, sample_weight=w_te),
        'Precisão (depressão)': precision_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'Recall (depressão)': recall_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'F1 (depressão)': f1_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
    }
    matriz = confusion_matrix(y_te, y_pred)

    # Importância agregada por variável original (soma das colunas One-Hot de cada uma)
    nomes = preprocessor.get_feature_names_out()
    importancias = pd.Series(modelo.feature_importances_, index=nomes)
    originais = sorted(X.columns, key=len, reverse=True)
    agregada = {}
    for nome_col, imp in importancias.items():
        base = nome_col.split('__', 1)[1]
        dono = next((c for c in originais if base == c or base.startswith(c + '_')), base)
        agregada[dono] = agregada.get(dono, 0) + imp
    importancia_var = pd.Series(agregada).sort_values(ascending=False)

    return metricas, matriz, importancia_var

resultados = {}

# 3. Cenário A: com variáveis de ansiedade (exploratório)

Removemos de $X$ somente os sintomas de depressão (PHQ). As variáveis de ansiedade (GAD-7) permanecem. Como ansiedade e depressão são fortemente comórbidas, este cenário mostra o **melhor desempenho que conseguimos quando há informação de saúde mental disponível**.

In [3]:
nome_a = 'A: com ansiedade (exploratório)'
resultados[nome_a] = treinar_avaliar(CENARIOS[nome_a])
metricas_a, matriz_a, imp_a = resultados[nome_a]

print(f'--- {nome_a} ---')
for k, v in metricas_a.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')
print('\nMatriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):')
print(matriz_a)
print('\nTop 10 variáveis mais importantes:')
print(imp_a.head(10).round(4).to_string())

--- A: com ansiedade (exploratório) ---
Nº de variáveis: 58
ROC-AUC: 0.9147
Acurácia: 0.8579
Precisão (depressão): 0.6475
Recall (depressão): 0.7525
F1 (depressão): 0.6961

Matriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):
[[3106  376]
 [ 228  707]]

Top 10 variáveis mais importantes:
TIPO_DIFICULDADE_COGNITIVA             0.0567
SEVERIDADE_ANSIEDADE                   0.0550
FREQ_ANSIEDADE                         0.0543
IDADE                                  0.0526
FREQ_ANSIEDADE_2SEM                    0.0502
FREQ_PREOCUPACAO_DESNECESSARIA_2SEM    0.0446
FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM    0.0445
FREQ_DIFICULDADE_DESCANSAR_2SEM        0.0398
FREQ_IRRITABILIDADE_2SEM               0.0330
FREQ_SOLITARIO                         0.0298


# 4. Cenário B: sem sintomas de saúde mental (hipótese do projeto)

Removemos de $X$ os sintomas de depressão **e** de ansiedade. O modelo passa a usar apenas variáveis sociodemográficas, de saúde física (doenças e condições crônicas), de acesso a serviços de saúde e de contexto social. É aqui que testamos se dá para **prever depressão a partir de doenças físicas e do contexto de vida**.

In [4]:
nome_b = 'B: sem sintomas mentais (hipótese)'
resultados[nome_b] = treinar_avaliar(CENARIOS[nome_b])
metricas_b, matriz_b, imp_b = resultados[nome_b]

print(f'--- {nome_b} ---')
for k, v in metricas_b.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')
print('\nMatriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):')
print(matriz_b)
print('\nTop 10 variáveis mais importantes:')
print(imp_b.head(10).round(4).to_string())

--- B: sem sintomas mentais (hipótese) ---
Nº de variáveis: 46
ROC-AUC: 0.8414
Acurácia: 0.8183
Precisão (depressão): 0.5774
Recall (depressão): 0.5967
F1 (depressão): 0.5868

Matriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):
[[3032  450]
 [ 352  583]]

Top 10 variáveis mais importantes:
IDADE                         0.0957
TIPO_DIFICULDADE_COGNITIVA    0.0853
FREQ_SOLITARIO                0.0692
SUPORTE_SOCIAL                0.0603
AUTOAVALIACAO_SAUDE_GERAL     0.0459
FREQUENCIA_FALHA_MEMORIA      0.0457
CATEGORIA_RAZAO_POBREZA       0.0441
DOR_CRONICA                   0.0437
QTD_DIFICULDADE_COGNITIVA     0.0423
RAZAO_RENDA_POBREZA           0.0349


# 5. Comparativo entre os cenários

A coluna **Variação (B − A)** mostra quanto desempenho se perde ao retirar a ansiedade. A coluna **Referência (acaso)** mostra o que um modelo sem nenhuma informação conseguiria: ROC-AUC de 0,5 e precisão igual à prevalência de depressão.

In [5]:
prevalencia = np.average(y_te, weights=w_te)

tabela = pd.DataFrame({nome: res[0] for nome, res in resultados.items()})
tabela['Variação (B − A)'] = tabela[nome_b] - tabela[nome_a]
tabela['Referência (acaso)'] = [np.nan, 0.5, np.nan, prevalencia, np.nan, np.nan]

print(f'Prevalência de depressão no teste (ponderada): {prevalencia:.1%}\n')
tabela.round(4)

Prevalência de depressão no teste (ponderada): 21.6%



,A: com ansiedade (exploratório),B: sem sintomas mentais (hipótese),Variação (B − A),Referência (acaso)
Nº de variáveis,58.0000,46.0000,-12.0000,NaN
ROC-AUC,0.9147,0.8414,-0.0732,0.5000
Acurácia,0.8579,0.8183,-0.0396,NaN
Precisão (depressão),0.6475,0.5774,-0.0702,0.2163
Recall (depressão),0.7525,0.5967,-0.1558,NaN
F1 (depressão),0.6961,0.5868,-0.1092,NaN


In [6]:
# Variáveis que o cenário B (sem sintomas mentais) usa para prever depressão
print('Top 15 variáveis do cenário B (importância agregada por variável original):\n')
print(imp_b.head(15).round(4).to_string())

ganho_auc = metricas_b['ROC-AUC'] - 0.5
perda_auc = metricas_a['ROC-AUC'] - metricas_b['ROC-AUC']
print(f"\nROC-AUC do cenário B acima do acaso: +{ganho_auc:.3f}")
print(f"ROC-AUC perdido ao retirar a ansiedade (A → B): -{perda_auc:.3f}")


Top 15 variáveis do cenário B (importância agregada por variável original):

IDADE                           0.0957
TIPO_DIFICULDADE_COGNITIVA      0.0853
FREQ_SOLITARIO                  0.0692
SUPORTE_SOCIAL                  0.0603
AUTOAVALIACAO_SAUDE_GERAL       0.0459
FREQUENCIA_FALHA_MEMORIA        0.0457
CATEGORIA_RAZAO_POBREZA         0.0441
DOR_CRONICA                     0.0437
QTD_DIFICULDADE_COGNITIVA       0.0423
RAZAO_RENDA_POBREZA             0.0349
ESCOLARIDADE_MAXIMA_FAMILIA     0.0348
LIMITACAO_TRABALHO_POR_SAUDE    0.0249
TEMPO_MORADIA_ANOS              0.0241
ESTRESSE_FINANCEIRO_MEDICO      0.0229
CATEGORIA_IMC                   0.0221

ROC-AUC do cenário B acima do acaso: +0.341
ROC-AUC perdido ao retirar a ansiedade (A → B): -0.073


# 6. Por que usar o alvo binário?

O PHQ original tem 4 níveis de severidade (1 = mínima, 2 = leve, 3 = moderada, 4 = grave). O alvo binário agrupa os níveis 2 a 4 como **com depressão** e deixa o nível 1 como **sem depressão**. Para justificar essa escolha, treinamos o mesmo Random Forest para prever os 4 níveis, nas mesmas linhas e no mesmo split, e comparamos.

In [7]:
y4 = df_modelo[TARGET_ORIGINAL]
y4_tr, y4_te = y4.iloc[pos_tr], y4.iloc[pos_te]

# Tamanho de cada nível: quantidade de registros e proporção ponderada pela amostra
distribuicao = pd.DataFrame({
    'Registros': y4.value_counts().sort_index(),
    '% ponderado': (df_modelo.groupby(TARGET_ORIGINAL)[PESO_COLUNA].sum() / df_modelo[PESO_COLUNA].sum() * 100).round(1),
})
distribuicao.index.name = 'Nível PHQ'
distribuicao

,Registros,% ponderado
Nível PHQ,,
1,17407,78.8
2,3139,14.2
3,1012,4.6
4,526,2.4


In [8]:
def preparar_matrizes(colunas_remover):
    X = df_modelo.drop(columns=BASE_REMOVER + colunas_remover)
    X_tr, X_te = X.iloc[pos_tr], X.iloc[pos_te]
    num_cols = [c for c in ['RAZAO_RENDA_POBREZA'] if c in X.columns]
    cat_cols = [c for c in X.columns if c not in num_cols]
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ])
    # Sem vazamento do teste: escalonador e One-Hot aprendem só no treino (fit) e apenas aplicam no teste (transform)
    return preprocessor.fit_transform(X_tr), preprocessor.transform(X_te)


def novo_rf(**parametros):
    base = dict(n_estimators=300, max_depth=25, min_samples_split=5, max_features='log2',
                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
    base.update(parametros)
    return RandomForestClassifier(**base)


def avaliar_multinivel(modelo, X_te_proc):
    y_pred = modelo.predict(X_te_proc)
    y_proba = modelo.predict_proba(X_te_proc)
    recall_nivel = recall_score(y4_te, y_pred, average=None, sample_weight=w_te, zero_division=0)
    resumo = {
        'ROC-AUC (OvR)': roc_auc_score(y4_te, y_proba, multi_class='ovr', sample_weight=w_te),
        'Acurácia': accuracy_score(y4_te, y_pred, sample_weight=w_te),
        'F1 macro': f1_score(y4_te, y_pred, average='macro', sample_weight=w_te, zero_division=0),
    }
    for nivel, r in zip(modelo.classes_, recall_nivel):
        resumo[f'Recall nível {nivel}'] = r
    return resumo, confusion_matrix(y4_te, y_pred)


matrizes = {nome: preparar_matrizes(remover) for nome, remover in CENARIOS.items()}
multinivel = {}
for nome, (X_tr_p, X_te_p) in matrizes.items():
    modelo = novo_rf().fit(X_tr_p, y4_tr)
    multinivel[nome] = avaliar_multinivel(modelo, X_te_p)

print('Matriz de confusão do cenário B, 4 níveis (nº de registros do teste; linhas = real, colunas = previsto):')
print(multinivel[nome_b][1])
pd.DataFrame({nome: res[0] for nome, res in multinivel.items()}).round(4)

Matriz de confusão do cenário B, 4 níveis (nº de registros do teste; linhas = real, colunas = previsto):
[[2956  496   22    8]
 [ 275  297   22   19]
 [  58  102   27   26]
 [  11   46   20   32]]


,A: com ansiedade (exploratório),B: sem sintomas mentais (hipótese)
ROC-AUC (OvR),0.9067,0.8298
Acurácia,0.7920,0.7525
F1 macro,0.5579,0.4463
Recall nível 1,0.8676,0.8599
Recall nível 2,0.5963,0.4585
Recall nível 3,0.2769,0.1255
Recall nível 4,0.5609,0.3077


### Resultado: os 4 níveis não são separáveis com os dados disponíveis

**1. Os níveis mais graves são raros.** Depois de remover registros incompletos, os níveis 3 e 4 somam 1.538 registros (cerca de 7% da população ponderada), contra 17.407 do nível 1. Com tão poucos exemplos, o modelo tem pouco para aprender sobre eles.

**2. O modelo erra sobretudo nos níveis intermediários.** No cenário B, o recall é de 86% no nível 1, mas só 46% no nível 2, **13% no nível 3** e 31% no nível 4. Na matriz de confusão, dos casos reais de nível 3 apenas 27 são acertados; a maioria vai para os níveis 1 e 2. O F1 macro, que dá o mesmo peso a cada nível, cai para 0,45 (0,56 no cenário B).

**3. A distinção entre níveis vizinhos é pouco informativa.** Os níveis são ordinais e a diferença entre "leve" e "moderado" depende de poucos pontos na escala. Variáveis de saúde física e de contexto social indicam bem **se há** sofrimento depressivo, mas não **quanto**.

**4. O alvo binário responde melhor à pergunta do projeto.** A hipótese é verificar se dá para **identificar** quem tem depressão sem sintomas mentais, e isso é uma pergunta de triagem (sim ou não). O binário tem classes mais equilibradas (cerca de 21% de positivos), métricas diretas (precisão, recall e ROC-AUC) e resultado fácil de interpretar. Mesmo assim, a ordenação por risco é preservada: o ROC-AUC do binário no cenário B (0,84) é equivalente ao do multinível (0,83 em média entre os níveis).

O multinível continua no notebook como **análise complementar**. Ele mostra até onde os dados permitem ir e quais níveis são mais difíceis de prever.

# 7. Ajuste de hiperparâmetros (cenário B)

Até aqui usamos hiperparâmetros fixos. Aqui buscamos uma configuração melhor **apenas no cenário B**, que é o que testa a hipótese, com `RandomizedSearchCV` (15 combinações, validação cruzada de 3 partes). O `PESO_AMOSTRAL` não é usado no treino nem na busca; ele só pondera as métricas de avaliação. O conjunto de teste não participa da busca.

In [9]:
from sklearn.model_selection import RandomizedSearchCV

ESPACO_BUSCA = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, 25, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', 0.3],
    'class_weight': ['balanced', 'balanced_subsample'],
}
X_tr_b, X_te_b = matrizes[nome_b]


def buscar(y_treino, scoring):
    busca = RandomizedSearchCV(
        RandomForestClassifier(random_state=RANDOM_STATE), ESPACO_BUSCA,
        n_iter=15, scoring=scoring, cv=3, random_state=RANDOM_STATE, n_jobs=-1,
    )
    # Hiperparâmetros escolhidos só com validação cruzada no treino (cv=3): o teste não participa da busca
    busca.fit(X_tr_b, y_treino)
    return busca


# Binário: seleciona pelo ROC-AUC
busca_bin = buscar(y_tr, 'roc_auc')
print('Melhores hiperparâmetros (binário):', busca_bin.best_params_)

modelo_bin = busca_bin.best_estimator_
y_pred = modelo_bin.predict(X_te_b)
y_proba = modelo_bin.predict_proba(X_te_b)[:, 1]
tuned_bin = {
    'ROC-AUC': roc_auc_score(y_te, y_proba, sample_weight=w_te),
    'Acurácia': accuracy_score(y_te, y_pred, sample_weight=w_te),
    'Precisão (depressão)': precision_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
    'Recall (depressão)': recall_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
    'F1 (depressão)': f1_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
}
comparacao_bin = pd.DataFrame({
    'B base': {k: resultados[nome_b][0][k] for k in tuned_bin},
    'B ajustado': tuned_bin,
})
comparacao_bin['Variação'] = comparacao_bin['B ajustado'] - comparacao_bin['B base']
print('\nMatriz de confusão do modelo ajustado (nº de registros do teste):')
print(confusion_matrix(y_te, y_pred))
comparacao_bin.round(4)

Melhores hiperparâmetros (binário): {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 25, 'class_weight': 'balanced'}

Matriz de confusão do modelo ajustado (nº de registros do teste):
[[3032  450]
 [ 352  583]]


,B base,B ajustado,Variação
ROC-AUC,0.8414,0.8414,0.0
Acurácia,0.8183,0.8183,0.0
Precisão (depressão),0.5774,0.5774,0.0
Recall (depressão),0.5967,0.5967,0.0
F1 (depressão),0.5868,0.5868,0.0


In [10]:
# Multinível (4 níveis): seleciona pelo F1 macro, que dá o mesmo peso a cada nível
busca_mult = buscar(y4_tr, 'f1_macro')
print('Melhores hiperparâmetros (4 níveis):', busca_mult.best_params_)

modelo_mult = busca_mult.best_estimator_
res_ajustado, matriz_ajustada = avaliar_multinivel(modelo_mult, X_te_b)
print('\nMatriz de confusão do modelo ajustado, 4 níveis (nº de registros do teste):')
print(matriz_ajustada)
comparacao_mult = pd.DataFrame({'B base': multinivel[nome_b][0], 'B ajustado': res_ajustado})
comparacao_mult['Variação'] = comparacao_mult['B ajustado'] - comparacao_mult['B base']
comparacao_mult.round(4)

Melhores hiperparâmetros (4 níveis): {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 25, 'class_weight': 'balanced'}

Matriz de confusão do modelo ajustado, 4 níveis (nº de registros do teste):
[[2956  496   22    8]
 [ 275  297   22   19]
 [  58  102   27   26]
 [  11   46   20   32]]


,B base,B ajustado,Variação
ROC-AUC (OvR),0.8298,0.8298,0.0
Acurácia,0.7525,0.7525,0.0
F1 macro,0.4463,0.4463,0.0
Recall nível 1,0.8599,0.8599,0.0
Recall nível 2,0.4585,0.4585,0.0
Recall nível 3,0.1255,0.1255,0.0
Recall nível 4,0.3077,0.3077,0.0


# 8. Perfil médio por nível de severidade

Para entender o que distingue os níveis do PHQ, comparamos a média (ponderada pela amostra) das variáveis mais importantes do cenário B em cada nível. As variáveis são códigos categóricos ordinais, então a média serve apenas para mostrar a **tendência** entre os níveis, e não um valor literal. Os valores `-1` (não resposta) já estão fora desta base.

In [11]:
top_b = [c for c in resultados[nome_b][2].head(8).index if c != 'RAZAO_RENDA_POBREZA']


def media_ponderada(grupo):
    return pd.Series({c: np.average(grupo[c], weights=grupo[PESO_COLUNA]) for c in top_b})


perfil = df_modelo.groupby(TARGET_ORIGINAL)[[*top_b, PESO_COLUNA]].apply(media_ponderada)
perfil.index.name = 'Nível PHQ'
perfil.round(2)

,IDADE,TIPO_DIFICULDADE_COGNITIVA,FREQ_SOLITARIO,SUPORTE_SOCIAL,AUTOAVALIACAO_SAUDE_GERAL,FREQUENCIA_FALHA_MEMORIA,CATEGORIA_RAZAO_POBREZA,DOR_CRONICA
Nível PHQ,,,,,,,,
1,48.96,3.66,1.61,4.38,3.77,3.61,10.16,1.79
2,46.24,3.17,2.38,3.97,3.18,2.93,9.16,2.39
3,43.76,2.97,2.82,3.59,2.96,2.69,8.62,2.61
4,46.77,2.89,3.41,3.28,2.36,2.51,7.31,3.04


# 9. Teste de sensibilidade: o resultado depende das variáveis cognitivas e de solidão?

No cenário B, `TIPO_DIFICULDADE_COGNITIVA` está entre as variáveis mais importantes. Ela vem da pergunta sobre dificuldade para **lembrar ou se concentrar**, e dificuldade de concentração também é um item do PHQ, que já removemos. Há o risco de essa variável reintroduzir sintoma depressivo no modelo e inflar o resultado. O mesmo cuidado vale para a solidão.

Para testar, repetimos o cenário B (mesmas linhas, mesmo split e mesmos hiperparâmetros) retirando progressivamente:

1. `TIPO_DIFICULDADE_COGNITIVA` sozinha;
2. o **bloco cognitivo** inteiro, que vem da mesma pergunta (`TIPO_DIFICULDADE_COGNITIVA`, `QTD_DIFICULDADE_COGNITIVA` e `FREQUENCIA_FALHA_MEMORIA`);
3. o bloco cognitivo **e** `FREQ_SOLITARIO`.

O teste de retirar só uma variável é fraco: as outras do bloco carregam a mesma informação e assumem o lugar dela. Por isso o passo 2 é o mais informativo.

In [12]:
BLOCO_COGNITIVO = ['TIPO_DIFICULDADE_COGNITIVA', 'QTD_DIFICULDADE_COGNITIVA', 'FREQUENCIA_FALHA_MEMORIA']
SOLIDAO = ['FREQ_SOLITARIO']
COLUNAS_B = CENARIOS[nome_b]

SENSIBILIDADE = {
    'B (referência)': COLUNAS_B,
    'B sem TIPO_DIFICULDADE_COGNITIVA': COLUNAS_B + BLOCO_COGNITIVO[:1],
    'B sem bloco cognitivo': COLUNAS_B + BLOCO_COGNITIVO,
    'B sem bloco cognitivo e sem solidão': COLUNAS_B + BLOCO_COGNITIVO + SOLIDAO,
}

res_sens = {nome: treinar_avaliar(remover) for nome, remover in SENSIBILIDADE.items()}

tabela_sens = pd.DataFrame({nome: res[0] for nome, res in res_sens.items()}).T
tabela_sens['Variação do ROC-AUC vs. B'] = tabela_sens['ROC-AUC'] - tabela_sens.loc['B (referência)', 'ROC-AUC']
tabela_sens['Nº de variáveis'] = tabela_sens['Nº de variáveis'].astype(int)
print(f'Referência de acaso: ROC-AUC 0,5 e precisão {prevalencia:.1%} (prevalência)\n')
print('Top 5 variáveis de cada cenário:')
for nome, res in res_sens.items():
    print(f'  {nome}: {", ".join(res[2].head(5).index)}')
tabela_sens.round(4)

Referência de acaso: ROC-AUC 0,5 e precisão 21.6% (prevalência)

Top 5 variáveis de cada cenário:
  B (referência): IDADE, TIPO_DIFICULDADE_COGNITIVA, FREQ_SOLITARIO, SUPORTE_SOCIAL, AUTOAVALIACAO_SAUDE_GERAL
  B sem TIPO_DIFICULDADE_COGNITIVA: IDADE, FREQ_SOLITARIO, SUPORTE_SOCIAL, QTD_DIFICULDADE_COGNITIVA, FREQUENCIA_FALHA_MEMORIA
  B sem bloco cognitivo: IDADE, FREQ_SOLITARIO, SUPORTE_SOCIAL, AUTOAVALIACAO_SAUDE_GERAL, DOR_CRONICA
  B sem bloco cognitivo e sem solidão: IDADE, SUPORTE_SOCIAL, AUTOAVALIACAO_SAUDE_GERAL, DOR_CRONICA, CATEGORIA_RAZAO_POBREZA


,Nº de variáveis,ROC-AUC,Acurácia,Precisão (depressão),Recall (depressão),F1 (depressão),Variação do ROC-AUC vs. B
B (referência),46,0.8414,0.8183,0.5774,0.5967,0.5868,0.0000
B sem TIPO_DIFICULDADE_COGNITIVA,45,0.8420,0.8178,0.5767,0.5923,0.5844,0.0006
B sem bloco cognitivo,43,0.8262,0.8172,0.5737,0.6030,0.5880,-0.0152
B sem bloco cognitivo e sem solidão,42,0.8038,0.7929,0.5210,0.5300,0.5254,-0.0377


### Resultado do teste de sensibilidade

| Cenário | Variáveis | ROC-AUC | Precisão | Recall | F1 |
|---|---|---|---|---|---|
| B (referência) | 46 | 0,841 | 57,7% | 59,7% | 0,587 |
| sem `TIPO_DIFICULDADE_COGNITIVA` | 45 | 0,842 | 57,7% | 59,2% | 0,584 |
| sem bloco cognitivo | 43 | 0,826 | 57,4% | 60,3% | 0,588 |
| sem bloco cognitivo e sem solidão | 42 | **0,804** | 52,1% | 53,0% | 0,525 |

- **Tirar só o `TIPO`:** não muda nada. `QTD_DIFICULDADE_COGNITIVA` e `FREQUENCIA_FALHA_MEMORIA` assumem o papel dela.
- **Tirar o bloco cognitivo:** o ROC-AUC cai apenas 0,015. Precisão, recall e F1 ficam praticamente iguais. Portanto o bloco cognitivo não é o que sustenta o resultado.
- **Tirar também a solidão:** o ROC-AUC cai para 0,804 e a precisão para 52,1%, ainda mais que o dobro da prevalência (21,6%). Nesse cenário, `DOR_CRONICA` aparece entre as 5 variáveis mais importantes, junto com idade, suporte social, autoavaliação de saúde e renda.
- **Leitura para a hipótese:** mesmo sem as variáveis mais próximas de sintomas depressivos, o modelo continua bem acima do acaso. Isso reforça que **saúde física, funcionamento e contexto social** têm informação real sobre depressão, e que o resultado do cenário B não vem apenas de um efeito de sobreposição com sintomas.
- **Limite do teste:** ele retira só as variáveis que identificamos como suspeitas. Outras variáveis (por exemplo, suporte social) também podem se relacionar com sintomas. O teste mostra robustez, mas não prova ausência total de sobreposição.

# 10. Priorizando o recall: rastreio em saúde

Em um rastreio de depressão, os dois erros não custam o mesmo. Um **falso negativo** é uma pessoa com depressão que o modelo não identifica e que, portanto, não é encaminhada para avaliação. Um **falso positivo** é uma pessoa sem depressão que passa por uma avaliação a mais. Por isso é preferível encontrar mais casos, mesmo que isso piore os falsos positivos. O Random Forest, com o limiar padrão de 0,5, é o modelo que menos casos encontra entre os de melhor desempenho (recall de 59,7% no cenário B). Esta seção mostra quanto recall dá para ganhar **baixando o limiar** de probabilidade e quanto isso custa: o modelo sinaliza mais pessoas, o recall sobe e a precisão cai.

**Como o limiar é escolhido, sem olhar o teste:** calculamos a probabilidade prevista dos registros de treino com validação cruzada de 5 partes (`cross_val_predict`) e, para cada **recall-alvo** (80%, 85%, 90% e 95%), usamos o limiar de probabilidade que alcança esse recall entre os casos de depressão do treino. Depois aplicamos o limiar ao teste e medimos o que realmente aconteceu. Por isso o recall obtido no teste pode ficar um pouco acima ou abaixo do alvo.

**Como ler a tabela:** `Recall` e `Precisão` são ponderados pelo `PESO_AMOSTRAL`, como nas seções anteriores. `TP`, `FN`, `FP` e `TN` são **contagens de registros** do teste, sem peso, e `% da amostra sinalizada` e `Sinalizados por caso encontrado` também usam contagens. Este último é quantas pessoas precisam ser avaliadas para achar um caso de depressão.

In [13]:
from sklearn.model_selection import cross_val_predict

ALVOS_RECALL = [0.80, 0.85, 0.90, 0.95]


def avaliar(y_proba, limiar=0.5):
    y_pred = (y_proba >= limiar).astype(int)
    metricas = {
        'ROC-AUC': roc_auc_score(y_te, y_proba, sample_weight=w_te),
        'Acurácia': accuracy_score(y_te, y_pred, sample_weight=w_te),
        'Precisão (depressão)': precision_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'Recall (depressão)': recall_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'F1 (depressão)': f1_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
    }
    return metricas, confusion_matrix(y_te, y_pred)


def analisar_limiares(colunas_remover, **parametros):
    """Escolhe o limiar no treino (validação cruzada) para cada recall-alvo e mede o resultado no teste."""
    X_tr_p, X_te_p = preparar_matrizes(colunas_remover)
    proba_te = novo_rf(**parametros).fit(X_tr_p, y_tr).predict_proba(X_te_p)[:, 1]
    # O limiar de cada recall-alvo vem das probabilidades de validação cruzada do TREINO; o teste só mede o recall obtido
    proba_cv = cross_val_predict(novo_rf(**parametros), X_tr_p, y_tr, cv=5, method='predict_proba')[:, 1]
    proba_positivos = proba_cv[(y_tr == 1).to_numpy()]

    linhas, matrizes = {}, {}
    for alvo in [None] + ALVOS_RECALL:
        limiar = 0.5 if alvo is None else np.quantile(proba_positivos, 1 - alvo)
        nome = 'Padrão (limiar 0,5)' if alvo is None else f'Recall-alvo {alvo:.0%}'
        metricas, matrizes[nome] = avaliar(proba_te, limiar)
        tn, fp, fn, tp = matrizes[nome].ravel()
        linhas[nome] = {
            'Limiar': limiar,
            'Recall': metricas['Recall (depressão)'],
            'Precisão': metricas['Precisão (depressão)'],
            'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn,
            '% da amostra sinalizada': (tp + fp) / len(y_te),
            'Sinalizados por caso encontrado': (tp + fp) / tp,
        }
    return pd.DataFrame.from_dict(linhas, orient='index'), matrizes

In [14]:
tabela_lim_b, matrizes_lim_b = analisar_limiares(COLUNAS_B, **busca_bin.best_params_)

print(f'Cenário B (modelo ajustado). Prevalência de depressão no teste (ponderada): {prevalencia:.1%}')
print('\nMatriz de confusão com recall-alvo de 90% (nº de registros do teste; linhas = real, colunas = previsto):')
print(matrizes_lim_b['Recall-alvo 90%'])
tabela_lim_b.round(3)

Cenário B (modelo ajustado). Prevalência de depressão no teste (ponderada): 21.6%

Matriz de confusão com recall-alvo de 90% (nº de registros do teste; linhas = real, colunas = previsto):
[[2037 1445]
 [  97  838]]


,Limiar,Recall,Precisão,TP,FN,FP,TN,% da amostra sinalizada,Sinalizados por caso encontrado
"Padrão (limiar 0,5)",0.500,0.597,0.577,583,352,450,3032,0.234,1.772
Recall-alvo 80%,0.371,0.792,0.467,747,188,928,2554,0.379,2.242
Recall-alvo 85%,0.330,0.835,0.424,786,149,1146,2336,0.437,2.458
Recall-alvo 90%,0.276,0.883,0.380,838,97,1445,2037,0.517,2.724
Recall-alvo 95%,0.202,0.934,0.314,880,55,2018,1464,0.656,3.293


### Resultado (cenário B)

- **Mais recall, mais falsos positivos.** Do limiar padrão (0,5) para o recall-alvo de 90%, o recall vai de 59,7% para **88,3%** e os casos perdidos caem de 352 para **97**. O custo são os falsos positivos, que mais que triplicam (de 450 para **1.445**), e a precisão, que cai de 57,7% para 38,0%.
- **Custo em avaliações.** Para cada caso encontrado, são avaliadas 1,8 pessoas no limiar padrão, 2,7 no recall-alvo de 90% e 3,3 no de 95%.
- **Continua muito melhor que o acaso.** No recall-alvo de 90%, o modelo sinaliza 51,7% da amostra e encontra 838 dos 935 casos. Um sorteio que sinalizasse a mesma fração encontraria cerca de 480. A precisão de 38,0% é 1,8 vez a prevalência (21,6%), e no alvo de 95% ela é 1,5 vez (31,4%).
- **O recall obtido no teste fica perto do alvo.** Ele ficou entre 0,8 e 1,7 ponto abaixo dos alvos (79,2%, 83,5%, 88,3% e 93,4%), porque o limiar vem do treino.
- **O limiar padrão é conservador para um rastreio.** Com 0,5, o Random Forest encontra só 59,7% dos casos, contra 72% a 73% de SVM e XGBoost. Com o limiar escolhido pelo recall-alvo (cerca de 0,28 para 90%), ele chega ao mesmo patamar. Qual alvo usar é uma decisão de saúde, não estatística: depende do custo de avaliar cada pessoa sinalizada e da capacidade do serviço. Se a prioridade é não deixar casos para trás, o alvo de 90% é um ponto de partida razoável: cerca de 4 em cada 10 pessoas sinalizadas têm depressão, e 97 dos 935 casos (10%) ficam sem identificação.

### Mesmo teste no cenário mais conservador

Repetimos a análise sem o bloco cognitivo e sem a solidão (seção 9), as variáveis mais próximas de sintomas depressivos.

In [15]:
tabela_lim_cons, _ = analisar_limiares(COLUNAS_B + BLOCO_COGNITIVO + SOLIDAO, **busca_bin.best_params_)
tabela_lim_cons.round(3)

,Limiar,Recall,Precisão,TP,FN,FP,TN,% da amostra sinalizada,Sinalizados por caso encontrado
"Padrão (limiar 0,5)",0.500,0.530,0.521,528,407,497,2985,0.232,1.941
Recall-alvo 80%,0.362,0.801,0.400,751,184,1188,2294,0.439,2.582
Recall-alvo 85%,0.320,0.849,0.359,795,140,1482,2000,0.516,2.864
Recall-alvo 90%,0.273,0.899,0.316,841,94,1892,1590,0.619,3.250
Recall-alvo 95%,0.219,0.948,0.275,889,46,2417,1065,0.748,3.719


- No limiar padrão, o cenário conservador encontra 528 dos 935 casos (recall de 53,0%, precisão de 52,1%). Para chegar a um recall próximo de 90% (**89,9%** no teste), o modelo sinaliza 61,9% da amostra, com 1.892 falsos positivos e precisão de **31,6%** (1,5 vez a prevalência).
- Portanto, **dá para atingir um recall alto mesmo sem as variáveis cognitivas e de solidão**, mas o custo é maior: cerca de 3,3 pessoas avaliadas por caso encontrado, contra 2,7 no cenário B completo.

# 11. O que mais dá para fazer pelo recall no Random Forest

Há dois tipos de alavanca para encontrar mais casos. **Deslocar o ponto de operação** (baixar o limiar, como na seção 10, ou dar mais peso à classe com depressão) aumenta o recall, mas também os falsos positivos, ao longo da mesma curva de precisão × recall. **Melhorar o modelo** sobe a curva: mais recall com os mesmos falsos positivos.

Para separar os dois, usamos duas medidas que **não dependem do limiar**: o **PR-AUC** e a **precisão com recall de 90%**, que é a maior precisão que o modelo alcança quando exigimos que ele encontre pelo menos 90% dos casos do teste. Se uma variante aumenta o recall no limiar padrão, mas deixa essas duas medidas iguais, ela só moveu o ponto de operação. Um modelo sem informação teria PR-AUC e precisão iguais à prevalência (21,6%).

Variantes testadas no cenário B, a partir da configuração ajustada da seção 7:

1. **Peso da classe positiva maior que o `balanced`** (que equivale a cerca de 3,7): pesos 6 e 10.
2. **Folhas maiores** (`min_samples_leaf = 20`, árvores sem limite de profundidade): probabilidades mais suaves, que respondem melhor a um limiar mais baixo.
3. **Extra Trees**, uma floresta com divisões sorteadas, que também suaviza as probabilidades.
4. **Subamostragem balanceada**: média de 10 florestas, cada uma treinada com todos os positivos e uma amostra de negativos do mesmo tamanho (a ideia da *Balanced Random Forest*, feita aqui só com `scikit-learn`).

In [16]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve


def precisao_no_recall(escore, recall_alvo=0.90):
    """Maior precisão (ponderada) que o modelo alcança no teste exigindo ao menos `recall_alvo` de recall.

    ATENÇÃO: o limiar é escolhido olhando o próprio teste (cenário ideal). Serve só para comparar modelos e é um
    limite superior otimista; o resultado realista é o da tabela de limiares escolhidos na validação cruzada."""
    precisao, recall, _ = precision_recall_curve(y_te, escore, sample_weight=w_te)
    return precisao[recall >= recall_alvo].max()


def resumir(escore, limiar=0.5):
    """Medidas que não dependem do limiar e o resultado no limiar padrão."""
    metricas, matriz = avaliar(escore, limiar)
    tn, fp, fn, tp = matriz.ravel()
    return {
        'ROC-AUC': metricas['ROC-AUC'],
        'PR-AUC': average_precision_score(y_te, escore, sample_weight=w_te),
        'Precisão com recall de 90%': precisao_no_recall(escore),
        'Recall': metricas['Recall (depressão)'],
        'Precisão': metricas['Precisão (depressão)'],
        'Casos perdidos (FN)': fn,
        'Falsos positivos (FP)': fp,
    }


def proba_rf_balanceado(n_modelos=10):
    """Média de Random Forests, cada um treinado com todos os positivos e uma amostra de negativos do mesmo tamanho."""
    rng = np.random.RandomState(RANDOM_STATE)
    y = y_tr.to_numpy()
    positivos, negativos = np.where(y == 1)[0], np.where(y == 0)[0]
    probas = []
    for i in range(n_modelos):
        idx = np.concatenate([positivos, rng.choice(negativos, size=len(positivos), replace=False)])
        rf = RandomForestClassifier(n_estimators=60, min_samples_leaf=5, max_features='sqrt',
                                    random_state=RANDOM_STATE + i, n_jobs=-1)
        probas.append(rf.fit(X_tr_b[idx], y[idx]).predict_proba(X_te_b)[:, 1])
    return np.mean(probas, axis=0)

In [17]:
VARIANTES_RF = {
    'Base (class_weight balanced)': {},
    'Peso da classe positiva 6': dict(class_weight={0: 1, 1: 6}),
    'Peso da classe positiva 10': dict(class_weight={0: 1, 1: 10}),
    'Folhas maiores (min_samples_leaf = 20)': dict(max_depth=None, min_samples_leaf=20, max_features='sqrt'),
}
escores = {
    nome: novo_rf(**{**busca_bin.best_params_, **ajuste}).fit(X_tr_b, y_tr).predict_proba(X_te_b)[:, 1]
    for nome, ajuste in VARIANTES_RF.items()
}

extra = ExtraTreesClassifier(n_estimators=300, min_samples_leaf=10, max_features='sqrt', class_weight='balanced',
                             random_state=RANDOM_STATE, n_jobs=-1)
escores['Extra Trees (min_samples_leaf = 10)'] = extra.fit(X_tr_b, y_tr).predict_proba(X_te_b)[:, 1]
escores['Subamostragem balanceada (10 florestas)'] = proba_rf_balanceado()

tabela_alavancas = pd.DataFrame.from_dict({nome: resumir(e) for nome, e in escores.items()}, orient='index')
print(f'Referência de acaso: PR-AUC e precisão com recall de 90% iguais à prevalência ({prevalencia:.1%})\n')
tabela_alavancas.round(4)

Referência de acaso: PR-AUC e precisão com recall de 90% iguais à prevalência (21.6%)



,ROC-AUC,PR-AUC,Precisão com recall de 90%,Recall,Precisão,Casos perdidos (FN),Falsos positivos (FP)
Base (class_weight balanced),0.8414,0.6341,0.3622,0.5967,0.5774,352,450
Peso da classe positiva 6,0.8414,0.6263,0.3732,0.7152,0.5188,254,691
Peso da classe positiva 10,0.8399,0.6252,0.3529,0.8195,0.4414,160,1077
Folhas maiores (min_samples_leaf = 20),0.8388,0.6226,0.3658,0.7166,0.4932,242,783
Extra Trees (min_samples_leaf = 10),0.8444,0.6354,0.3693,0.7035,0.5214,263,677
Subamostragem balanceada (10 florestas),0.8417,0.6280,0.3644,0.7489,0.4933,223,812


### Resultado

- **O peso da classe positiva resolve o recall baixo do limiar padrão.** Com peso 6, o recall vai de 59,7% para 71,5% e os casos perdidos caem de 352 para 254. Com peso 10, o recall é de cerca de 82% (160 perdidos). O custo são os falsos positivos, que sobem de 450 para 691 e 1.077. Com peso 6, o Random Forest chega ao patamar de SVM e XGBoost no limiar padrão (72% a 73% de recall e precisão de 52%). Mas o ROC-AUC (0,840 a 0,841) e a precisão com recall de 90% (0,353 a 0,373) ficam na mesma faixa: é o mesmo modelo operando em outro ponto da curva, como ao baixar o limiar (seção 10).
- **Folhas maiores, Extra Trees e subamostragem balanceada também elevam o recall no limiar padrão** (de 70% a 75%), porque suavizam as probabilidades ou equilibram as classes, mas não sobem a curva: o PR-AUC fica entre 0,623 e 0,635 (0,634 na base) e a precisão com recall de 90% entre 0,364 e 0,369 (0,362 na base).
- **O maior ROC-AUC é o do Extra Trees (0,844)**, 0,003 acima da base, uma diferença dentro da variação de um único teste.
- **Conclusão para o Random Forest:** o recall baixo era uma questão de ponto de operação, e não do modelo. Dois caminhos equivalentes resolvem: um peso maior para a classe positiva ou o limiar da seção 10. Nenhuma variante testada sobe a curva, então cada caso a mais custa falsos positivos. O limite está na informação das variáveis, não no algoritmo.

# 12. Interpretação e conclusão

## Binário: a hipótese se sustenta, com desempenho menor que o do cenário A

| Métrica | A (com ansiedade) | B (sem sintomas mentais) | Acaso |
|---|---|---|---|
| ROC-AUC | 0,915 | **0,841** | 0,50 |
| Precisão (depressão) | 64,8% | **57,7%** | 21,6% (prevalência) |
| Recall (depressão) | 75,3% | **59,7%** | - |
| F1 (depressão) | 0,696 | **0,587** | - |

- Retirar as variáveis de ansiedade custa cerca de 0,07 de ROC-AUC e 16 pontos de recall. Isso era esperado, já que ansiedade e depressão são fortemente comórbidas (no cenário A, 8 das 10 variáveis mais importantes são de ansiedade ou cognição).
- Mesmo assim, **sem nenhum sintoma de saúde mental, o modelo fica 0,34 de ROC-AUC acima do acaso e a precisão é 2,7 vezes a prevalência** (57,7% contra 21,6%). Ou seja, dados físicos, funcionais e de contexto social **carregam informação real sobre depressão**.
- As variáveis mais importantes no cenário B são idade, dificuldade cognitiva, solidão, suporte social, autoavaliação de saúde, falha de memória, renda/pobreza e dor crônica. O sinal físico vem sobretudo de dor, limitações e autoavaliação de saúde, e não do diagnóstico de doenças clássicas. Isso é coerente com o estudo de associação em `02 Análise exploratória/07_associacao_depressao_variaveis_nao_mentais_2025.ipynb`.
- **Ressalva:** solidão, memória e dificuldade cognitiva podem se sobrepor aos próprios sintomas depressivos. Para sustentar a tese "dados físicos", vale destacar dor crônica, limitações funcionais e autoavaliação de saúde.

## Multinível: informativo, mas pior que o binário para o objetivo do projeto

- Com 4 níveis, o cenário B tem ROC-AUC (OvR) de 0,830 e F1 macro de 0,446. O recall é bom no nível 1 (86%) e fraco nos níveis 3 e 4 (13% e 31%), por causa da raridade desses níveis e da dificuldade de separar níveis vizinhos (seção 6).

## Ajuste de hiperparâmetros

- **Binário:** a busca escolheu exatamente a configuração que já usávamos (300 árvores, profundidade 25, `log2`), então não houve ganho. Isso indica que o **limite está na informação disponível nas variáveis, não nos hiperparâmetros**.
- **Multinível:** a busca também escolheu a configuração base, sem ganho.

## Teste de sensibilidade (seção 9)

- Retirar o bloco cognitivo (tipo e quantidade de dificuldade e frequência de falha de memória) reduz o ROC-AUC do cenário B de 0,841 para 0,826. Retirar também a solidão leva a 0,804, ainda bem acima do acaso (0,5).
- Portanto, a hipótese **não depende** das variáveis que mais se confundem com sintomas depressivos. Sem elas, os destaques passam a ser idade, suporte social, autoavaliação de saúde, renda e dor crônica.

## Recall: rastreio em saúde (seções 10 e 11)

**Pontos de operação no cenário B** (modelos ajustados). Recall e precisão são ponderados pelo `PESO_AMOSTRAL` e os falsos positivos são contagens do teste. Cada modelo tem um limiar padrão diferente. No recall-alvo de 90%, o limiar vem do treino:

| Modelo | Limiar padrão: recall e precisão | Recall-alvo de 90%: recall obtido | Precisão | Falsos positivos | % da amostra sinalizada |
|---|---|---|---|---|---|
| SVM linear | 72,4% e 52,7% | 86,9% | 38,5% | 1.341 | 48,9% |
| XGBoost | 73,2% e 52,1% | 88,4% | 38,8% | 1.393 | 50,5% |
| Random Forest | 59,7% e 57,7% | 88,3% | 38,0% | 1.445 | 51,7% |
| KNN com colunas ponderadas | 66,8% e 52,9% | 88,0% | 37,0% | 1.530 | 53,7% |
| KNN ajustado (seção 4) | 45,6% e 60,8% | 87,2% | 34,0% | 1.679 | 56,8% |

**O que mais melhora o recall em cada algoritmo** (seção 11 deste notebook e seções equivalentes dos outros):

| Algoritmo | Alavanca testada | Recall no limiar padrão | Melhora a curva de precisão × recall? |
|---|---|---|---|
| Random Forest | peso 6 para a classe positiva | 59,7% → 71,5% | Não: só desloca o ponto de operação |
| KNN | colunas ponderadas pelo SVM L1 | 45,6% → 66,8% | **Sim:** PR-AUC 0,600 → 0,646 e ROC-AUC 0,816 → 0,843 |
| SVM linear | peso 5 e 8 para a classe positiva | 72,4% → 78,8% e 84,7% | Não |
| XGBoost | `scale_pos_weight` × 2 | 73,2% → 85,5% | Não |

- **O limiar padrão é o ponto fraco do Random Forest para rastreio.** Com 0,5, o modelo encontra 59,7% dos casos no cenário B. Com o recall-alvo de 90% (limiar de cerca de 0,28, escolhido no treino), ele encontra 838 dos 935 casos e deixa 97 sem identificação (352 no limiar padrão), ao custo de 1.445 falsos positivos (450 no limiar padrão) e de uma precisão de 38,0%, ainda 1,8 vez a prevalência. Mesmo no cenário conservador (sem cognição e sem solidão), um recall de 90% é alcançável, com precisão de 31,6%.
- **No mesmo recall, o Random Forest se aproxima de SVM e XGBoost.** No alvo de 90%, a precisão é de 38,0%, contra 38,5% e 38,8%, e os falsos positivos são 1.445, contra 1.341 e 1.393, uma diferença dentro da variação de um único teste. O ROC-AUC do Random Forest é um pouco menor (0,841 contra 0,856). O KNN, mesmo com colunas ponderadas, precisa de mais falsos positivos (1.530).
- **Nenhuma variante do Random Forest sobe a curva.** O peso 6 para a classe positiva leva o recall no limiar padrão a 71,5%, o patamar de SVM e XGBoost, mas só desloca o ponto de operação. Folhas maiores, Extra Trees e subamostragem balanceada também não melhoram o PR-AUC (0,623 a 0,635) nem a precisão com recall de 90% (0,353 a 0,373). Entre os quatro algoritmos, só no KNN mudar o modelo ajudou: ponderar as colunas pelos coeficientes de um SVM L1 subiu o PR-AUC de 0,600 para 0,646.

## Limitações

- O resultado vale para registros completos (22.084 de 24.215); quem não respondeu a alguma pergunta ficou de fora.
- Métricas ponderadas pela amostra, com um único split treino/teste. Uma validação cruzada daria uma estimativa mais estável.
- O recall-alvo (80% a 95%) é uma decisão de saúde, não estatística. O notebook mostra o custo de cada alvo, mas não define qual usar. O limiar é escolhido no treino, e o recall obtido no teste varia alguns pontos em torno do alvo.
- As variantes da seção 11 foram comparadas em um único teste, com cerca de 900 casos positivos. Diferenças de um a dois pontos entre variantes ficam dentro dessa variação.
- Os modelos mostram **associação preditiva**, não causalidade.